In [4]:
import os
import time
import re
from googletrans import Translator

# --- CONFIG ---
INPUT_FOLDER = "italy_txts"
OUTPUT_FOLDER = "italy_translated_txts"
MAX_CHARS = 2000 # Smaller chunks to avoid malformed URL errors

def clean_text(text):
    # This removes strings of dots often used in Italian indices (Indice)
    # and common non-standard character clusters found in OCR-processed texts
    text = re.sub(r'\.{2,}', ' ', text)
    text = re.sub(r'[o*]{3,}', ' ', text)
    return text

def run_translation():
    translator = Translator()
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    
    # Identify all source files
    all_files = [f for f in os.listdir(INPUT_FOLDER) if f.endswith('.txt')]
    # Filter out those already present in the output folder
    files_to_process = [f for f in all_files if not os.path.exists(os.path.join(OUTPUT_FOLDER, f))]
    
    total_files = len(all_files)
    already_done = total_files - len(files_to_process)
    
    print(f"Progress Status: {already_done}/{total_files} files already translated.")
    print(f"Starting translation for {len(files_to_process)} remaining files...\n")

    for index, filename in enumerate(files_to_process, start=already_done + 1):
        output_path = os.path.join(OUTPUT_FOLDER, filename)
        
        # Start timer for the current file
        file_start_time = time.time()
        print(f"[{index}/{total_files}] Processing: {filename}")
        
        try:
            with open(os.path.join(INPUT_FOLDER, filename), 'r', encoding='utf-8') as f:
                content = f.read()
            
            content = clean_text(content)
            chunks = [content[i:i+MAX_CHARS] for i in range(0, len(content), MAX_CHARS)]
            
            translated_parts = []
            
            for idx, chunk in enumerate(chunks, 1):
                try:
                    time.sleep(1) # Reduced delay slightly for better efficiency
                    res = translator.translate(chunk, src='it', dest='en')
                    translated_parts.append(res.text)
                except AttributeError as e:
                    if "as_dict" in str(e):
                        print(f"    [!] Part {idx} contains 'Toxic' characters. Keeping original.")
                        translated_parts.append(f"\n[SECTION UNTRANSLATABLE - KEEPING ORIGINAL]\n{chunk}\n")
                    else:
                        raise e
                except Exception as e:
                    print(f"    [!] Unexpected error on Part {idx}: {e}")
                    translated_parts.append(chunk)

            with open(output_path, 'w', encoding='utf-8') as f:
                f.write("\n\n".join(translated_parts))
            
            # Calculate elapsed time
            elapsed_time = time.time() - file_start_time
            print(f"    [✓] Saved. (Time: {elapsed_time:.2f}s)")

        except Exception as e:
            print(f"    [X] Critical failure on {filename}: {e}")

if __name__ == "__main__":
    run_translation()

Progress Status: 7885/7885 files already translated.
Starting translation for 0 remaining files...

